In [ ]:
import numpy as np

import matplotlib.pyplot as plt
from qgravnet import QGravNetFactory
from sklearn.metrics import roc_auc_score, roc_curve

from hls4ml_gravnet.utils.data import load_processed, shuffle_vertices
from hls4ml_gravnet.utils.evaluation import load_run, response_rmse
from hls4ml_gravnet.utils.files import get_project_root_dir

PROJECT_ROOT = get_project_root_dir("hls4ml-gravnet")

In [ ]:
RUN = "mini_128V_L1_S3"
model_cfg, weights_path, history, datapath, n_vertices, is_shuffled = load_run(PROJECT_ROOT / "data/results/" / RUN)
D = load_processed(datapath)
if 'gravnet_kwargs' in model_cfg:
    model_cfg['gravnet_cfg'] = model_cfg.pop('gravnet_kwargs')
trained_model = QGravNetFactory(**model_cfg).create_keras_model(n_vertices, 4)
trained_model.load_weights(weights_path)
# trained_model.summary()

if is_shuffled:
    D["X_hits_test"] = shuffle_vertices(D["X_hits_test"], seed=0)
D["X_hits_test"] = D["X_hits_test"][:, :n_vertices, :]
test_energy_pred, test_pid_pred = trained_model.predict(D["X_hits_test"])

In [ ]:
test_response_rmse = response_rmse(D["y_energy_test"], test_energy_pred)
test_auc = roc_auc_score(D["y_pid_test"], test_pid_pred)
fpr, tpr, thresholds = roc_curve(D["y_pid_test"], test_pid_pred)

print(f"Test Response RMSE: {test_response_rmse:.4f}")
print(f"Test PID AUC: {test_auc:.4f}")

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(fpr, tpr)
plt.xlabel("Pion False Positive Rate")
plt.ylabel("Pion True Positive Rate")
plt.xlim(0.0, 1.0)
plt.ylim(0.0, 1.0)

plt.subplot(1, 2, 2)
plt.hist(
    test_energy_pred.flatten() / D["y_energy_test"],
    bins=50,
    # histtype="stepfilled",
    alpha=0.7,
    density=True,
)
plt.axvline(1.0, color="k", linestyle="--", lw=1, alpha=0.7)
plt.xlabel("Predicted / True Energy")
plt.ylabel("Density")
plt.xlim(0.0, 4.0)

spcr = " " * 5
notes_dataset = "Trained on small Garnet dataset \n(1 file, 10k events)" if "mini" in datapath else "Trained on full Garnet dataset \n(50 files, à 10k events)"
notes_sdim = f"Learned coordinate space dimensionality: {model_cfg['n_dimensions']}"
notes = f"{notes_dataset}\n\n{n_vertices} vertices\n\nNo large skip connections\n{notes_sdim}\n"
descr = (
    "QGravNet Evaluation \n\n"
    + spcr
    + f"AUC: {test_auc:.3f} \n"
    + spcr
    + f"Response RMS: {test_response_rmse:.3f}"
    # + "\n\n\nModel Config (changes from default):\n\n"
    # + "".join([f"{spcr}{k}: {v}\n" for k, v in model_cfg.items()])
    + "\n\nNotes: \n\n"
    + notes
)
plt.text(
    1.05,
    1.0,
    descr,
    transform=plt.gca().transAxes,
    fontsize=11,
    verticalalignment="top",
    horizontalalignment="left",
)

plt.tight_layout()

plt.savefig(f"/scratch/lasfour/hgcal-clustering/hls4ml-gravnet/data/results/{RUN}/eval_plot.png", dpi=300)
plt.savefig(f"/scratch/lasfour/hgcal-clustering/hls4ml-gravnet/data/results/{RUN}/eval_plot.pdf")
plt.show()